# 🔬 Phase 1: Data Preparation & Synthetic Data Generation
**Đề tài:** Nghiên cứu xây dựng hệ thống trích xuất và cấu trúc hóa thông tin tự động từ văn bản hành chính tiếng Việt sử dụng LLMs và OCR

**Notebook này thực hiện:**
1. ⚙️ Cài đặt môi trường
2. 🔴 Trích xuất con dấu đỏ từ 150 PDF test
3. 🔴 Tạo 200+ synthetic stamps giống thật
4. 📄 Chuyển 2000 docx → Image + Overlay stamps → Training pairs cho GAN
5. 🧠 Tạo instruction dataset cho LLM fine-tuning (từ 2000 docx)

> **Yêu cầu:** Upload folder `data/` lên Google Drive trước khi chạy

## ⚙️ Cell 1: Cài đặt packages

In [1]:
!apt-get install -y libreoffice
!pip install -q python-docx Pillow opencv-python-headless numpy pandas PyMuPDF tqdm

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  apparmor at-spi2-core default-jre default-jre-headless dictionaries-common
  firebird3.0-common firebird3.0-common-doc firebird3.0-server-core
  firebird3.0-utils fonts-crosextra-caladea fonts-crosextra-carlito
  fonts-dejavu fonts-dejavu-core fonts-dejavu-extra fonts-liberation2
  fonts-linuxlibertine fonts-noto-core fonts-noto-extra fonts-noto-mono
  fonts-noto-ui-core fonts-opensymbol fonts-sil-gentium
  fonts-sil-gentium-basic gsettings-desktop-schemas gstreamer1.0-gl
  gstreamer1.0-gtk3 gstreamer1.0-plugins-base hunspell-en-us libabsl20210324
  libabw-0.1-1 libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libboost-filesystem1.74.0
  libboost-iostreams1.74.0 libboost-locale1.74.0 libboost-thread1.74.0
  libbsh-java libcdparanoia0 libcdr-0.1-1 libclucene-contribs1v5
  libclucen

In [7]:

# Update package lists and install LibreOffice with fix-missing flag
!apt-get update
!apt-get install -y libreoffice --fix-missing

# Verify installation
!libreoffice --version


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntu

## 📂 Cell 2: Mount Google Drive & Cấu hình đường dẫn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import cv2
import json
import random
import math
import re
import zipfile
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from tqdm import tqdm

# ====== CẤU HÌNH ĐƯỜNG DẪN ======
BASE_DIR = "/content/drive/MyDrive/OCR-LLM_Research"
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DOCX_DIR = os.path.join(DATA_DIR, "raw_word_files")
STAMPS_MANUAL_DIR = os.path.join(DATA_DIR, "stamps")
# Thư mục chứa dấu đã xử lý sạch
STAMPS_PROCESSED_DIR = os.path.join(DATA_DIR, "stamps_processed")

CLEAN_IMAGES_DIR = os.path.join(DATA_DIR, "processed/clean_images")
STAMPED_IMAGES_DIR = os.path.join(DATA_DIR, "processed/stamped_images")
LLM_TRAINING_DIR = os.path.join(DATA_DIR, "llm_training")

for d in [CLEAN_IMAGES_DIR, STAMPED_IMAGES_DIR, LLM_TRAINING_DIR, STAMPS_PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Đã thiết lập cấu hình và thư mục xử lý con dấu.")

Mounted at /content/drive
✅ Đã thiết lập cấu hình và thư mục xử lý con dấu.


---
## 📄 Cell 3: Chuyển DOCX → Clean Image + Stamped Image
Tạo training pairs cho Pix2Pix GAN: `{ảnh sạch}` ↔ `{ảnh có dấu}`

In [3]:
import subprocess
import fitz
import random

import os
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image

def only_remove_background(stamp_path, output_path):
    """
    High-end Stamp Extraction: Tách nền, chuẩn hóa màu và làm nét chuyên nghiệp.
    """
    img = cv2.imread(stamp_path)
    if img is None: return False

    # 1. Khử nhiễu nhẹ để làm mượt nét vẽ
    denoised = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)

    # 2. Chuyển sang LAB để tách kênh màu đỏ tốt hơn
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # Kênh 'a' đại diện cho dải màu từ xanh lá đến đỏ. Giá trị cao = đỏ đậm.
    _, mask = cv2.threshold(a, 145, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3. Morphological operations để lấp đầy các lỗ nhỏ trong con dấu
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    # 4. Chuẩn hóa màu đỏ (Làm rõ nét)
    # Tăng độ tương phản cho phần màu đỏ
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l_enhanced = clahe.apply(l)

    # Hợp nhất lại và chuyển về BGR
    enhanced_lab = cv2.merge([l_enhanced, a, b])
    result_bgr = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

    # 5. Tạo ảnh RGBA với mask đã xử lý
    b_channel, g_channel, r_channel = cv2.split(result_bgr)
    alpha_channel = mask

    # Làm mịn biên của alpha channel để tránh răng cưa
    alpha_channel = cv2.GaussianBlur(alpha_channel, (3, 3), 0)

    result_rgba = cv2.merge((b_channel, g_channel, r_channel, alpha_channel))

    cv2.imwrite(output_path, result_rgba)
    return True

def overlay_stamp(clean_path, stamp_path, output_path):
    clean = Image.open(clean_path).convert('RGBA')
    stamp = Image.open(stamp_path).convert('RGBA')
    max_w = int(clean.width * random.uniform(0.18, 0.28))
    scale = max_w / max(stamp.width, 1)
    stamp = stamp.resize((int(stamp.width * scale), int(stamp.height * scale)), Image.LANCZOS)

    # Xoay nhẹ con dấu để trông thật hơn
    stamp = stamp.rotate(random.uniform(-5, 5), expand=True, resample=Image.BICUBIC)

    x = clean.width - stamp.width - random.randint(100, 300)
    y = clean.height - stamp.height - random.randint(200, 500)

    # Trộn con dấu với độ trong suốt ngẫu nhiên nhẹ để giả lập mực in
    clean.paste(stamp, (max(0, x), max(0, y)), stamp)
    clean.convert('RGB').save(output_path, 'PNG')

def convert_docx_to_pdf(docx_path, out_dir):
    try:
        cmd = ['libreoffice', '--headless', '--convert-to', 'pdf', '--outdir', out_dir, docx_path]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return os.path.join(out_dir, os.path.splitext(os.path.basename(docx_path))[0] + ".pdf")
    except: return None

def process_document_perfectly(pdf_path, category, filename, cleaned_stamps, dpi=300):
    count = 0
    try:
        doc = fitz.open(pdf_path)
        for p_idx in range(len(doc)):
            page = doc[p_idx]
            pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
            clean_name = f"{category}_{filename}_p{p_idx}.png"
            clean_p = os.path.join(CLEAN_IMAGES_DIR, clean_name)
            stamped_p = os.path.join(STAMPED_IMAGES_DIR, clean_name)
            pix.save(clean_p)

            chance = 0.95 if p_idx == len(doc)-1 else (0.30 if p_idx == 0 else 0.10)
            if random.random() < chance and cleaned_stamps:
                overlay_stamp(clean_p, random.choice(cleaned_stamps), stamped_p)
            else:
                Image.open(clean_p).save(stamped_p)
            count += 1
        doc.close()
    except: pass
    return count

# --- THỰC THI QUY TRÌNH ---
docx_files = []
for category in os.listdir(RAW_DOCX_DIR):
    cat_dir = os.path.join(RAW_DOCX_DIR, category)
    if os.path.isdir(cat_dir):
        for f in os.listdir(cat_dir):
            if f.endswith('.docx'): docx_files.append((os.path.join(cat_dir, f), category))

PAIR_LIMIT = 2000

print("1️⃣ Bước 1: Tách nền & Chuẩn hóa con dấu (Professional Extraction)...")
for s_file in tqdm(os.listdir(STAMPS_MANUAL_DIR)):
    if s_file.endswith(('.png', '.jpg', '.jpeg')):
        only_remove_background(os.path.join(STAMPS_MANUAL_DIR, s_file), os.path.join(STAMPS_PROCESSED_DIR, s_file.split('.')[0] + '.png'))

cleaned_stamps = [os.path.join(STAMPS_PROCESSED_DIR, f) for f in os.listdir(STAMPS_PROCESSED_DIR)]
tmp_pdf_dir = "/content/tmp_pdf"
os.makedirs(tmp_pdf_dir, exist_ok=True)

print(f"2️⃣ Bước 2: Tạo bộ dữ liệu training chất lượng cao...")
total = 0
for docx_path, category in tqdm(docx_files[:PAIR_LIMIT], desc="Processing Docs"):
    pdf_p = convert_docx_to_pdf(docx_path, tmp_pdf_dir)
    if pdf_p and os.path.exists(pdf_p):
        total += process_document_perfectly(pdf_p, category, os.path.splitext(os.path.basename(docx_path))[0], cleaned_stamps)
        os.remove(pdf_p)

print(f"\n✅ Hoàn tất! Đã tạo {total} ảnh với con dấu được xử lý hoàn hảo.")

1️⃣ Bước 1: Tách nền & Chuẩn hóa con dấu (Professional Extraction)...


100%|██████████| 200/200 [00:43<00:00,  4.60it/s]


2️⃣ Bước 2: Tạo bộ dữ liệu training chất lượng cao...


Processing Docs: 100%|██████████| 2000/2000 [00:01<00:00, 1454.34it/s]


✅ Hoàn tất! Đã tạo 0 ảnh với con dấu được xử lý hoàn hảo.


In [8]:
import os
import subprocess
import fitz

def convert_docx_to_clean_images(docx_path, output_dir, dpi=300):
    """
    Chuyển đổi file DOCX thành các file ảnh sạch (clean images) tương ứng với từng trang.
    """
    # 1. Tạo thư mục tạm để chứa PDF
    temp_pdf_dir = "/content/tmp_conversion"
    os.makedirs(temp_pdf_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    try:
        # 2. Chuyển DOCX -> PDF sử dụng LibreOffice headless
        filename = os.path.splitext(os.path.basename(docx_path))[0]
        cmd = [
            'libreoffice', '--headless', '--convert-to', 'pdf',
            '--outdir', temp_pdf_dir, docx_path
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

        pdf_path = os.path.join(temp_pdf_dir, f"{filename}.pdf")

        # 3. Trích xuất ảnh từ PDF
        doc = fitz.open(pdf_path)
        saved_paths = []

        for p_idx in range(len(doc)):
            page = doc[p_idx]
            # Thiết lập độ phân giải (DPI)
            pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))

            clean_name = f"{filename}_page_{p_idx}.png"
            clean_path = os.path.join(output_dir, clean_name)

            pix.save(clean_path)
            saved_paths.append(clean_path)

        doc.close()
        # Xóa file PDF tạm
        if os.path.exists(pdf_path): os.remove(pdf_path)

        print(f"✅ Đã chuyển đổi xong: {len(saved_paths)} trang từ {filename}")
        return saved_paths

    except Exception as e:
        print(f"❌ Lỗi khi xử lý {docx_path}: {e}")
        return []

# --- Ví dụ sử dụng ---
# sample_docx = "/content/drive/MyDrive/OCR-LLM_Research/data/raw_word_files/CV/CV_001.docx"
# output_folder = "/content/drive/MyDrive/OCR-LLM_Research/data/processed/clean_images"
# convert_docx_to_clean_images(sample_docx, output_folder)

print(f"🚀 Bắt đầu chuyển đổi {min(len(docx_files), PAIR_LIMIT)} file DOCX sang Clean Images...")

total_pages_created = 0
# Sử dụng danh sách docx_files đã được khởi tạo ở các cell trước
for docx_path, category in tqdm(docx_files[:PAIR_LIMIT], desc="Batch Converting"):
    # Gọi hàm đã tạo để chuyển đổi
    saved_paths = convert_docx_to_clean_images(docx_path, CLEAN_IMAGES_DIR)
    total_pages_created += len(saved_paths)

print(f"\n✅ Hoàn tất! Tổng cộng đã tạo {total_pages_created} ảnh sạch trong thư mục: {CLEAN_IMAGES_DIR}")

🚀 Bắt đầu chuyển đổi 2000 file DOCX sang Clean Images...


Batch Converting:   0%|          | 1/2000 [00:02<1:29:44,  2.69s/it]

✅ Đã chuyển đổi xong: 1 trang từ QD_004


Batch Converting:   0%|          | 2/2000 [00:04<1:19:20,  2.38s/it]

✅ Đã chuyển đổi xong: 1 trang từ QD_003


...

✅ Đã chuyển đổi xong: 1 trang từ K_200


Batch Converting: 100%|█████████▉| 1999/2000 [1:18:38<00:02,  2.25s/it]

✅ Đã chuyển đổi xong: 1 trang từ K_199


Batch Converting: 100%|██████████| 2000/2000 [1:18:41<00:00,  2.36s/it]

✅ Đã chuyển đổi xong: 1 trang từ K_196

✅ Hoàn tất! Tổng cộng đã tạo 2200 ảnh sạch trong thư mục: /content/drive/MyDrive/OCR-LLM_Research/data/processed/clean_images


### 👁️ Xem thử training pairs (Clean vs Stamped)

In [ ]:
clean_files = sorted([f for f in os.listdir(CLEAN_IMAGES_DIR) if f.endswith('.png')])[:4]
fig, axes = plt.subplots(2, 4, figsize=(20, 14))
for idx, f in enumerate(clean_files):
    clean_img = Image.open(os.path.join(CLEAN_IMAGES_DIR, f))
    stamped_path = os.path.join(STAMPED_IMAGES_DIR, f)
    # Crop phần dưới (có stamp) để zoom
    h = clean_img.height
    crop_box = (0, h//2, clean_img.width, h)
    axes[0][idx].imshow(clean_img.crop(crop_box))
    axes[0][idx].set_title(f'Clean (bottom half)', fontsize=9)
    axes[0][idx].axis('off')
    if os.path.exists(stamped_path):
        stamped_img = Image.open(stamped_path)
        axes[1][idx].imshow(stamped_img.crop(crop_box))
        axes[1][idx].set_title(f'With Stamp', fontsize=9)
    axes[1][idx].axis('off')
plt.suptitle('Training Pairs: Clean vs Stamped (bottom half)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 🧠 Cell 4: Tạo LLM Training Dataset (Instruction-Following)
Parse 2000 docx → trích xuất metadata → tạo Alpaca-style dataset cho fine-tuning Qwen-7B

In [ ]:
def extract_metadata(docx_path, category):
    """Trích xuất metadata từ docx bằng regex."""
    text = docx_to_text(docx_path)
    if not text or len(text) < 30:
        return None, None

    cat_map = {'CV': 'Công văn', 'HD': 'Hợp đồng', 'QD': 'Quy định',
               'TT': 'Tờ trình', 'K': 'Khác'}

    metadata = {
        "loai_van_ban": cat_map.get(category, "Khác"),
        "so_hieu": "", "ngay_ban_hanh": "",
        "trich_yeu": "", "nguoi_ky": "", "co_quan_ban_hanh": ""
    }

    # Số hiệu
    m = re.search(r'[Ss]ố[:\s]+(\d+[\/-][A-ZĐa-zđ\d\/-]+)', text)
    if m: metadata['so_hieu'] = m.group(1).strip()

    # Ngày
    m = re.search(r'[Nn]gày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text)
    if m:
        d, mo, y = m.groups()
        metadata['ngay_ban_hanh'] = f"{d.zfill(2)}/{mo.zfill(2)}/{y}"
    else:
        m = re.search(r'(\d{1,2})[\/-](\d{1,2})[\/-](\d{4})', text)
        if m:
            d, mo, y = m.groups()
            metadata['ngay_ban_hanh'] = f"{d.zfill(2)}/{mo.zfill(2)}/{y}"

    # Trích yếu
    m = re.search(r'[Vv]\/[Vv][:\s]+(.+?)(?:\n|$)', text)
    if m: metadata['trich_yeu'] = m.group(1).strip()[:200]

    # Người ký (heuristic: tên người ở cuối văn bản)
    lines = text.strip().split('\n')
    for line in reversed(lines[-10:]):
        line = line.strip()
        if line and len(line) < 50:
            words = line.split()
            if 2 <= len(words) <= 5 and all(w[0].isupper() for w in words if w):
                if not any(kw in line.lower() for kw in ['điều', 'khoản', 'mục', 'số']):
                    metadata['nguoi_ky'] = line
                    break

    return text, metadata


# ====== TẠO DATASET ======
CLASSIFICATION_INSTRUCTION = (
    "Bạn là chuyên gia phân loại văn bản hành chính Việt Nam. "
    "Hãy phân loại văn bản sau vào một trong các loại: "
    "Công văn, Hợp đồng, Quy định, Tờ trình, Khác. "
    "Chỉ trả lời tên loại văn bản, không giải thích thêm."
)

EXTRACTION_INSTRUCTION = (
    "Bạn là chuyên gia trích xuất thông tin từ văn bản hành chính Việt Nam. "
    "Hãy đọc văn bản sau và trích xuất các thông tin theo định dạng JSON:\n"
    "{\n"
    '  \"loai_van_ban\": \"<Công văn|Hợp đồng|Quy định|Tờ trình|Khác>\",\n'
    '  \"so_hieu\": \"<số hiệu văn bản>\",\n'
    '  \"ngay_ban_hanh\": \"<DD/MM/YYYY>\",\n'
    '  \"co_quan_ban_hanh\": \"<tên cơ quan>\",\n'
    '  \"trich_yeu\": \"<trích yếu nội dung>\",\n'
    '  \"nguoi_ky\": \"<họ tên người ký>\"\n'
    "}\n"
    "Nếu không tìm thấy thông tin, để trống (\"\")."
)

cat_map = {'CV': 'Công văn', 'HD': 'Hợp đồng', 'QD': 'Quy định',
           'TT': 'Tờ trình', 'K': 'Khác'}

dataset = []
print("📊 Đang tạo LLM instruction dataset từ 2000 docx files...")

for docx_path, category in tqdm(docx_files, desc="Building dataset"):
    text, metadata = extract_metadata(docx_path, category)
    if not text or len(text) < 30:
        continue

    # Truncate nếu quá dài
    input_text = text[:3000] if len(text) > 3000 else text

    # Task 1: Classification
    dataset.append({
        "instruction": CLASSIFICATION_INSTRUCTION,
        "input": input_text,
        "output": cat_map.get(category, "Khác")
    })

    # Task 2: Extraction
    if metadata:
        dataset.append({
            "instruction": EXTRACTION_INSTRUCTION,
            "input": input_text,
            "output": json.dumps(metadata, ensure_ascii=False, indent=2)
        })

# Shuffle & Split 80/10/10
random.shuffle(dataset)
n = len(dataset)
splits = {
    'train': dataset[:int(n*0.8)],
    'val': dataset[int(n*0.8):int(n*0.9)],
    'test': dataset[int(n*0.9):]
}

for name, data in splits.items():
    path = os.path.join(LLM_TRAINING_DIR, f"{name}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  💾 {name}: {len(data)} samples → {path}")

print(f"\n✅ Tổng: {len(dataset)} instruction samples")

### 📊 Thống kê dataset

In [ ]:
# Xem mẫu dataset
print("=" * 60)
print("📊 THỐNG KÊ DỮ LIỆU ĐÃ TẠO")
print("=" * 60)

ext_count = len([f for f in os.listdir(STAMPS_EXTRACTED_DIR) if f.endswith('.png')])
syn_count = len([f for f in os.listdir(STAMPS_SYNTHETIC_DIR) if f.endswith('.png')])
clean_count = len([f for f in os.listdir(CLEAN_IMAGES_DIR) if f.endswith('.png')])
stamped_count = len([f for f in os.listdir(STAMPED_IMAGES_DIR) if f.endswith('.png')])

print(f"🔴 Stamps extracted:  {ext_count}")
print(f"🔴 Stamps synthetic:  {syn_count}")
print(f"📄 Clean images:      {clean_count}")
print(f"📄 Stamped images:    {stamped_count}")

for split in ['train', 'val', 'test']:
    p = os.path.join(LLM_TRAINING_DIR, f"{split}.json")
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            d = json.load(f)
        print(f"🧠 LLM {split}:          {len(d)} samples")

# Xem 1 mẫu
print("\n" + "=" * 60)
print("📝 MẪU DATASET (extraction task):")
print("=" * 60)
sample = [s for s in dataset if '{' in s.get('output', '')][:1]
if sample:
    print(f"Input (first 200 chars): {sample[0]['input'][:200]}...")
    print(f"Output: {sample[0]['output']}")

---
## ✅ Phase 1 hoàn tất!
**Tiếp theo:** Chạy `Phase2_Stamp_Removal_GAN.ipynb` để train GAN xóa dấu.

**Dữ liệu đã tạo:**
- `data/processed/clean_images/` - Ảnh sạch
- `data/processed/stamped_images/` - Ảnh có overlay stamp
- `data/llm_training/` - train.json, val.json, test.json